# MindGrow: Adaptive Cognitive Difficulty Recommendation
### MCA Minor Project – Machine Learning Module
**Algorithm:** Random Forest Classifier  
**Objective:** Dynamically assess learner performance across three cognitive games (Memory, Attention, Pattern Recognition) and recommend an optimal next difficulty level (Easy, Medium, Hard).

---
### Features Used for Prediction:
1. game_encoded: Game Type (Memory=0, Attention=1, Pattern=2)
2. current_difficulty_encoded: Current Level (Easy=0, Medium=1, Hard=2)
3. score: Game points accumulated
4. ccuracy: Percentage accuracy (0 - 100%)
5. 	ime_taken: Total duration in seconds
6. mistakes: Number of erroneous attempts or missed items


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import joblib

# Set styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')


## 1. Load Training Dataset
We use synthetic benchmark data generated according to psychometric and cognitive learning curves to train our model. Actual user gameplay sessions are recorded separately in Data/game_history.csv.


In [ ]:
# Load cognitive training dataset
df = pd.read_csv('../Data/synthetic_cognitive_training.csv')
print(f'Total samples: {len(df)}')
df.head(10)


## 2. Data Preprocessing & Feature Encoding
Convert categorical attributes into integer representations suitable for the Random Forest Classifier.


In [ ]:
game_map = {'Memory': 0, 'Attention': 1, 'Pattern': 2}
diff_map = {'Easy': 0, 'Medium': 1, 'Hard': 2}
rev_diff_map = {0: 'Easy', 1: 'Medium', 2: 'Hard'}

df['game_encoded'] = df['game'].map(game_map)
df['current_difficulty_encoded'] = df['current_difficulty'].map(diff_map)
df['target_encoded'] = df['recommended_difficulty'].map(diff_map)

feature_cols = ['game_encoded', 'current_difficulty_encoded', 'score', 'accuracy', 'time_taken', 'mistakes']
X = df[feature_cols]
y = df['target_encoded']

# Split into 80% train and 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Training samples: {len(X_train)}, Testing samples: {len(X_test)}')


## 3. Model Training: Random Forest Classifier
We train a Random Forest ensemble with 100 decision trees to prevent overfitting and capture non-linear relationships between response time, mistake frequency, and accuracy.


In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=7,
    min_samples_split=4,
    random_state=42
)

rf_model.fit(X_train, y_train)
print('Random Forest Classifier successfully trained!')


## 4. Model Evaluation
Evaluating model accuracy, precision, recall, and F1-score on unseen test data.


In [ ]:
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Test Set Accuracy: {accuracy * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Easy', 'Medium', 'Hard']))


## 5. Feature Importance Analysis
Visualizing which cognitive metrics most heavily drive the model's difficulty recommendation.


In [ ]:
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)

plt.figure(figsize=(9, 4.5))
colors = ['#6748D5', '#35C6C8', '#FF6B6B', '#FFC857', '#252344', '#8C68EE']
importances.plot(kind='barh', color=colors[:len(importances)])
plt.title('Random Forest Feature Importance for Difficulty Recommendation', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.ylabel('Cognitive Features')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


## 6. Export Model Bundle for Streamlit Application
We save the model bundle via Joblib so it can be loaded directly in the Streamlit application for real-time adaptive difficulty recommendations during gameplay.


In [ ]:
model_bundle = {
    'model': rf_model,
    'feature_cols': feature_cols,
    'game_map': game_map,
    'diff_map': diff_map,
    'rev_diff_map': rev_diff_map,
    'accuracy': accuracy,
    'classes': ['Easy', 'Medium', 'Hard']
}

joblib.dump(model_bundle, 'rf_difficulty_model.joblib')
print('Model exported to rf_difficulty_model.joblib successfully!')
